<h1>Chapter 5 | Data Exercise #5 | <code>World Development Indicators</code> | Generalizing from Data</h1>


<h2>Introduction:</h2>
<p>In this notebook, you will find my notes and code for Chapter 5's <b>exercise 5</b> of the book <a href="https://gabors-data-analysis.com/">Data Analysis for Business, Economics, and Policy</a>, by Gábor Békés and Gábor Kézdi. The question was: 

<p>5. Download the most recent data from the world development indicators website on GDP per capita and CO2 emission per capita.
<p>Assignments:</p>
<ul>
    <li>Divide countries into two groups by their GDP per capita and calculate the average difference in CO2 emission per capita between the two groups.</li>
    <li>Use bootstrap to estimate its standard error. </li>
    <li>Create the appropriate 95% CI.</li>
    <li>Interpret the results.</li>
</ul>
<h2>1. Load the data</h2>

In [97]:
import os
import pandas as pd
import warnings
from datetime import datetime
from plotnine import *
import sys
import numpy as np
from scipy.stats import norm, sem
from numpy.random import choice
import wbgapi as wb
warnings.filterwarnings("ignore")

In [98]:
# Increase number of returned rows in pandas
pd.set_option("display.max_rows", 500)

In [99]:
# Current script folder
dirname = os.getcwd()

# Get location folders

data_out = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/05-WDI_CO2_GDP/data/clean/"
output = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/05-WDI_CO2_GDP/data/output/"
func = f"{dirname}/da_case_studies/ch00-tech_prep/"
sys.path.append(func)
paths = [data_out, output]

for path in paths:
    if not os.path.exists(path):
        os.makedirs(path)

In [100]:
# Import the prewritten helper functions 
from py_helper_functions import *

## 1.1 Download the most recent data from the world development indicators website on GDP per capita and CO2 emission per capita.
While we could use the World Bank's APIs, there are several tools developed by others that already did the heavy lifting and can make our life easier. `wbgapi` is one among many. Let's learn a bit about it and use it in our favor.

For now, let's play around with it.

In [101]:
help(wb)

Help on package wbgapi:

NAME
    wbgapi

DESCRIPTION
    wbgapi provides a comprehensive interface to the World Bank's data and
    metadata API with built-in pandas integration

PACKAGE CONTENTS
    __version__
    data
    economy
    economy_coder
    economy_metadata
    income
    lending
    region
    series
    series_metadata
    source
    time
    topic
    utils

CLASSES
    builtins.Exception(builtins.BaseException)
        APIError
            APIResponseError
        URLError
    builtins.dict(builtins.object)
        Coder
    builtins.object
        Featureset
        Metadata
        MetadataCollection

    class APIError(builtins.Exception)
     |  APIError(url, msg, code=None)
     |
     |  Method resolution order:
     |      APIError
     |      builtins.Exception
     |      builtins.BaseException
     |      builtins.object
     |
     |  Methods defined here:
     |
     |  __init__(self, url, msg, code=None)
     |      Initialize self.  See help(type(self))

In [102]:
help(wb.series)

Help on module wbgapi.series in wbgapi:

NAME
    wbgapi.series - Access information about series in a database

FUNCTIONS
    Series(id='all', q=None, topic=None, db=None, name='SeriesName')
        Return a pandas Series by calling list

    get(id, db=None)
        Retrieve a specific series object

        Arguments:
            id:     the series identifier

            db:     database; pass None to access the global database

        Returns:
            a series object

        Example:
            print(wbgapi.series.get('SP.POP.TOTL')['value'])

    info(id='all', q=None, topic=None, db=None)
        Print a user report of series. This can be time consuming
        for large databases like the WDI if 'all' series are requested.

        Arguments:
            id:         a series identifier or list-like of identifiers

            q:          search string (on series name))

            topic:      topic ID or list-like

            db:         database; pass None to access t

In [103]:
wb.source.info()

id,name,code,concepts,lastupdated
1,Doing Business,DBS,3,2021-08-18
2,World Development Indicators,WDI,3,2025-12-16
3,Worldwide Governance Indicators,WGI,3,2024-11-05
5,Subnational Malnutrition Database,SNM,3,2016-03-21
6,International Debt Statistics,IDS,4,2025-12-03
11,Africa Development Indicators,ADI,3,2013-02-22
12,Education Statistics,EDS,3,2024-06-25
13,Enterprise Surveys,ESY,3,2022-03-25
14,Gender Statistics,GDS,3,2025-11-11
15,Global Economic Monitor,GEM,3,2025-12-17


In [104]:
wb.series.info(db=2)

id,value
AG.CON.FERT.PT.ZS,Fertilizer consumption (% of fertilizer production)
AG.CON.FERT.ZS,Fertilizer consumption (kilograms per hectare of arable land)
AG.LND.AGRI.K2,Agricultural land (sq. km)
AG.LND.AGRI.ZS,Agricultural land (% of land area)
AG.LND.ARBL.HA,Arable land (hectares)
AG.LND.ARBL.HA.PC,Arable land (hectares per person)
AG.LND.ARBL.ZS,Arable land (% of land area)
AG.LND.CREL.HA,Land under cereal production (hectares)
AG.LND.CROP.ZS,Permanent cropland (% of land area)
AG.LND.EL5M.RU.K2,Rural land area where elevation is below 5 meters (sq. km)


Fortunately, this awesome package has a search string parameter to help us. Let's try it out.

In [105]:
wb.series.info(q="GDP per capita")

id,value
NY.GDP.PCAP.CD,GDP per capita (current US$)
NY.GDP.PCAP.CN,GDP per capita (current LCU)
NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
NY.GDP.PCAP.KD.ZG,GDP per capita growth (annual %)
NY.GDP.PCAP.KN,GDP per capita (constant LCU)
NY.GDP.PCAP.PP.CD,"GDP per capita, PPP (current international $)"
NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 international $)"
,7 elements


In [106]:
# 2. Search for CO2 emissions
wb.series.info(db=None, q="CO2")

id,value
EN.GHG.CO2.AG.MT.CE.AR5,Carbon dioxide (CO2) emissions from Agriculture (Mt CO2e)
EN.GHG.CO2.BU.MT.CE.AR5,Carbon dioxide (CO2) emissions from Building (Energy) (Mt CO2e)
EN.GHG.CO2.FE.MT.CE.AR5,Carbon dioxide (CO2) emissions from Fugitive Emissions (Energy) (Mt CO2e)
EN.GHG.CO2.IC.MT.CE.AR5,Carbon dioxide (CO2) emissions from Industrial Combustion (Energy) (Mt CO2e)
EN.GHG.CO2.IP.MT.CE.AR5,Carbon dioxide (CO2) emissions from Industrial Processes (Mt CO2e)
EN.GHG.CO2.LU.DF.MT.CE.AR5,Carbon dioxide (CO2) net fluxes from LULUCF - Deforestation (Mt CO2e)
EN.GHG.CO2.LU.FL.MT.CE.AR5,Carbon dioxide (CO2) net fluxes from LULUCF - Forest Land (Mt CO2e)
EN.GHG.CO2.LU.MT.CE.AR5,Carbon dioxide (CO2) net fluxes from LULUCF - Total excluding non-tropical fires (Mt CO2e)
EN.GHG.CO2.LU.OL.MT.CE.AR5,Carbon dioxide (CO2) net fluxes from LULUCF - Other Land (Mt CO2e)
EN.GHG.CO2.LU.OS.MT.CE.AR5,Carbon dioxide (CO2) net fluxes from LULUCF - Organic Soil (Mt CO2e)


In [107]:
for row in wb.data.fetch("EN.GHG.CO2.PC.CE.AR5", "BRA"):
    print(row)

{'value': 2.31826466114939, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2024'}
{'value': 2.27447826989363, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2023'}
{'value': 2.27889577215227, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2022'}
{'value': 2.41580858865319, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2021'}
{'value': 2.1466461829, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2020'}
{'value': 2.25869881785082, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2019'}
{'value': 2.28562350357953, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2018'}
{'value': 2.41694955353585, 'series': 'EN.GHG.CO2.PC.CE.AR5', 'economy': 'BRA', 'aggregate': False, 'time': 'YR2017'}
{'value': 2.37533549789759, 'series': 'EN.GHG.CO2.PC.CE.AR5'

In [108]:
wb.data.DataFrame(["EN.GHG.CO2.PC.CE.AR5", "NY.GDP.PCAP.PP.CD"], "BRA", mrv=10)

,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
series,,,,,,,,,,
EN.GHG.CO2.PC.CE.AR5,2.568189,2.375335,2.41695,2.285624,2.258699,2.146646,2.415809,2.278896,2.274478,2.318265
NY.GDP.PCAP.PP.CD,14821.438318,14309.206149,14559.04889,15463.742639,16069.838015,16101.618306,18075.706005,19876.853353,21175.618699,22338.476564


Great! We've identified our series. We need to work with:
- `NY.GDP.PCAP.PP.CD` –	GDP per capita, PPP (current international $)
- `EN.GHG.CO2.PC.CE.AR5` – Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)

### 1.1.1 Fetch and Clean Data
First, we'll define our indicators. We map the complex codes to simpler names.

In [109]:
indicators = {
    "NY.GDP.PCAP.PP.CD": "GDP_PPP",
    "EN.GHG.CO2.PC.CE.AR5": "CO2_Capita"
}

Then, we can fetch data for the last 10 years. That should suffice.

In [110]:
df_raw = wb.data.DataFrame(indicators,
                           economy="all",
                           time=range(2014, 2025),
                           skipAggs=True, # Remove regional/world aggregates
                           numericTimeKeys=True, # Return years as integers
                           labels=True, # Add Country Column name
                           columns="series"
                           )

In [111]:
df_raw.head()

Country  Time  EN.GHG.CO2.PC.CE.AR5  NY.GDP.PCAP.PP.CD
economy time                                                         
ZWE     2024  Zimbabwe  2024              0.772821        5928.080625
        2023  Zimbabwe  2023              0.768994        5791.192129
        2022  Zimbabwe  2022              0.719221        5395.856298
        2021  Zimbabwe  2021              0.688925        4827.088694
        2020  Zimbabwe  2020              0.584444        4178.585579

In [112]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2387 entries, ('ZWE', np.int64(2024)) to ('AFG', np.int64(2014))
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country               2387 non-null   object 
 1   Time                  2387 non-null   object 
 2   EN.GHG.CO2.PC.CE.AR5  2233 non-null   float64
 3   NY.GDP.PCAP.PP.CD     2175 non-null   float64
dtypes: float64(2), object(2)
memory usage: 156.4+ KB


Awesome! Now, we need to clean the index/columns.

In [113]:
df = df_raw.reset_index()
df

,economy,time,Country,Time,EN.GHG.CO2.PC.CE.AR5,NY.GDP.PCAP.PP.CD
0,ZWE,2024,Zimbabwe,2024,0.772821,5928.080625
1,ZWE,2023,Zimbabwe,2023,0.768994,5791.192129
2,ZWE,2022,Zimbabwe,2022,0.719221,5395.856298
3,ZWE,2021,Zimbabwe,2021,0.688925,4827.088694
4,ZWE,2020,Zimbabwe,2020,0.584444,4178.585579
...,...,...,...,...,...,...
2382,AFG,2018,Afghanistan,2018,0.321419,2432.276701
2383,AFG,2017,Afghanistan,2017,0.291328,2335.795862
2384,AFG,2016,Afghanistan,2016,0.219353,2213.181441
2385,AFG,2015,Afghanistan,2015,0.249091,2284.075848


Our dictionary values were not passed as column names. Let's try to fix that as well.

In [114]:
df = df.rename(columns=indicators)

In [115]:
df

,economy,time,Country,Time,CO2_Capita,GDP_PPP
0,ZWE,2024,Zimbabwe,2024,0.772821,5928.080625
1,ZWE,2023,Zimbabwe,2023,0.768994,5791.192129
2,ZWE,2022,Zimbabwe,2022,0.719221,5395.856298
3,ZWE,2021,Zimbabwe,2021,0.688925,4827.088694
4,ZWE,2020,Zimbabwe,2020,0.584444,4178.585579
...,...,...,...,...,...,...
2382,AFG,2018,Afghanistan,2018,0.321419,2432.276701
2383,AFG,2017,Afghanistan,2017,0.291328,2335.795862
2384,AFG,2016,Afghanistan,2016,0.219353,2213.181441
2385,AFG,2015,Afghanistan,2015,0.249091,2284.075848


We can now drop wors where GDP or CO2 is missing.


In [116]:
df.isnull().sum()

economy         0
time            0
Country         0
Time            0
CO2_Capita    154
GDP_PPP       212
dtype: int64

In [117]:
df_complete = df.dropna(subset=["GDP_PPP", "CO2_Capita"])
df_complete.sample(5)

,economy,time,Country,Time,CO2_Capita,GDP_PPP
1614,GAB,2016,Gabon,2016,3.122878,13997.735284
448,SOM,2016,"Somalia, Fed. Rep.",2016,0.058388,1468.504385
2044,BDI,2015,Burundi,2015,0.039375,722.032898
2147,BMU,2022,Bermuda,2022,3.976895,105142.409967
1406,IND,2015,India,2015,1.715503,5425.035991


Now, we can sort our DataFrame. Later on, we'll keep the most recent year per country.

In [118]:
df_final = df_complete.sort_values(["Country", "time"], ascending=[True, False])
df_final.head()

,economy,time,Country,Time,CO2_Capita,GDP_PPP
2377,AFG,2023,Afghanistan,2023,0.280995,2201.722907
2378,AFG,2022,Afghanistan,2022,0.278421,2122.995815
2379,AFG,2021,Afghanistan,2021,0.312157,2144.166570
2380,AFG,2020,Afghanistan,2020,0.310825,2561.981761
2381,AFG,2019,Afghanistan,2019,0.319753,2583.485332


In [119]:
df_final = df_final.rename(columns={
    "economy": "Country_ISO",
    "Time": "Year"})

In [120]:
df_final = df_final.drop(columns="time").reset_index(drop=True)
df_final

,Country_ISO,Country,Year,CO2_Capita,GDP_PPP
0,AFG,Afghanistan,2023,0.280995,2201.722907
1,AFG,Afghanistan,2022,0.278421,2122.995815
2,AFG,Afghanistan,2021,0.312157,2144.166570
3,AFG,Afghanistan,2020,0.310825,2561.981761
4,AFG,Afghanistan,2019,0.319753,2583.485332
...,...,...,...,...,...
2081,ZWE,Zimbabwe,2018,0.816285,3992.890719
2082,ZWE,Zimbabwe,2017,0.714789,10756.445442
2083,ZWE,Zimbabwe,2016,0.771813,4275.449351
2084,ZWE,Zimbabwe,2015,0.881442,4045.898647


### 1.1.2 Keep Latest Data
For our analysis, we'll want to work with the latest data available for each country only.

In [121]:
df_final = df_final.drop_duplicates(subset="Country_ISO", keep="first")
df_final

,Country_ISO,Country,Year,CO2_Capita,GDP_PPP
0,AFG,Afghanistan,2023,0.280995,2201.722907
10,ALB,Albania,2024,1.785221,26693.193437
21,DZA,Algeria,2024,3.980800,17620.742396
32,AGO,Angola,2024,0.741712,10118.608995
43,ATG,Antigua and Barbuda,2024,3.798575,33386.191713
54,ARG,Argentina,2024,4.010665,30431.193122
65,ARM,Armenia,2024,2.470282,22823.179526
76,ABW,Aruba,2024,5.148386,50649.301523
87,AUS,Australia,2024,14.097354,71410.419875
98,AUT,Austria,2024,6.322784,71621.803364


In [123]:
print(f"Columns fixed: {df_final.columns.tolist()}")
print(df_final.head())

Columns fixed: ['Country_ISO', 'Country', 'Year', 'CO2_Capita', 'GDP_PPP']
   Country_ISO              Country  Year  CO2_Capita       GDP_PPP
0          AFG          Afghanistan  2023    0.280995   2201.722907
10         ALB              Albania  2024    1.785221  26693.193437
21         DZA              Algeria  2024    3.980800  17620.742396
32         AGO               Angola  2024    0.741712  10118.608995
43         ATG  Antigua and Barbuda  2024    3.798575  33386.191713


Awesome! We can move on to our next step.
# 2. Divide countries into two groups by their GDP per capita and calculate the average difference in CO2 emission per capita between the two groups.
## 2.1 Grouping countries by GDP
First, we need to define a cutoff line to divide our countries. We can use the median GDP.

In [124]:
median_gdp  = df_final["GDP_PPP"].median()
print(f"The cutoff line (Median GDP) is : ${median_gdp:.2f}")

The cutoff line (Median GDP) is : $19873.89


Ok! Now we can create our groups based on this median.

In [125]:
# Use .copy() to avoid warnings
df_final = df_final.copy()
df_final["Group"] = np.where(df_final["GDP_PPP"] > median_gdp, "Rich", "Poor")


In [126]:
df_final.sample(10)

,Country_ISO,Country,Year,CO2_Capita,GDP_PPP,Group
975,KAZ,Kazakhstan,2024,12.567819,40890.931769,Rich
1227,MUS,Mauritius,2024,3.536342,31839.759754,Rich
920,ISR,Israel,2024,5.781340,55690.739103,Rich
953,JPN,Japan,2024,7.842421,51685.038753,Rich
175,BEL,Belgium,2024,7.306615,72236.905055,Rich
1946,TUV,Tuvalu,2023,0.000000,6151.010470,Poor
2033,VUT,Vanuatu,2024,0.869799,3605.607367,Poor
1881,TON,Tonga,2023,1.546889,7802.669862,Poor
1348,NLD,Netherlands,2024,6.601884,84221.978378,Rich
43,ATG,Antigua and Barbuda,2024,3.798575,33386.191713,Rich


## 2.1 Calculate the average difference in CO2 emission per capita between the two groups

In [128]:
group_stats = df_final.groupby(by="Group")["CO2_Capita"].mean()
group_stats

Group
Poor    2.210134
Rich    6.847820
Name: CO2_Capita, dtype: float64

We can see that the average CO2 emissions per capita is quite different between the two groups. Let's see the average difference now.

In [129]:
actual_diff = group_stats["Rich"] - group_stats["Poor"]
print(f"Rich countries emit {actual_diff:.2f} tons more.")

Rich countries emit 4.64 tons more.
